In [2]:
from conllu_test import check_conllu_for_conditions2, head_greater_than_id, head_smaller_than_id, has_son, file_to_conllu,merge_evals, conllu_merger, check_conllu_for_conditions_v3, no_weak_pronoun, has_sons, next_head, prev_head, has_no_son, check_conds, feats_include, is_sub_clause_head_core_arg, nicht, is_not, is_not_any_of, has_no_sons, is_sub_clause_head, parent_is, merge_conditions_dicts
import os
from pathlib import Path
import datetime
import json
from pathlib import Path
import pandas as pd
import time

from analysis_main import conllu_file_stats, file_freqs_eval, file_freqs_eval_filtered_by_conds


In [3]:
# #FUNCTION 1: CHECK ONE FILE FOR SEVERAL CONDITIONS, return 1 dict of results
# # conditions are stored in a dictionary of conditions, with a name for the condition as key
# def eval_conds(conds_list, file):
#     results={"file": file }
#     for name, condition in conds_list.items():
#         results.update( {name :check_conllu_for_conditions2(file, condition, printf=False)} )
#     print(results)
#     return results

# FUNCTION 1 :input dict of conditions
# max ex refers to the maximum of examples it will include in the summary
#output: list of condition dictionaries, ready to json load
def eval_conds(d_conds, file, max_ex=100, custom_id= False):
    list=[]
    t0 = time.time()
    for name, cond in d_conds.items():
        t1=time.time()
        print(f"\tEvaluating {name} on {file}")
        results={"filename": file, "eval_series":0, "cond_name": name,  "matches":check_conllu_for_conditions2(file, cond, printf=False), "cond_content": cond, "examples":check_conllu_for_conditions_v3(file, cond, max_l=max_ex, custom_id=custom_id), "eval_date": datetime.datetime.now()}
        for key in results:
            results[key] = str(results[key])
        list.append(results)
        print(f"\t \tEvaluated {name} on {file}, it took {time.time()-t1}")
    print(f"done with list, it took {time.time()-t0}")
    return list


# FUNCITON 2: input: list of result dictionaries + name of the file the results should be printed in
# checks that the keys are the ones htat shoud be too
# If return_res is false, the dict gets printed in a json file, else it is returned
def print_eval_log(jlist, outfile, return_res=True):
    # check if the folder exists, else create it
    Path("eval/").mkdir(parents=True, exist_ok=True)

    outpath = "eval/"+outfile
    standard_dict = {"filename":0, "cond_name":"", "cond_content":0, "matches":0, "eval_date":0 , "examples": [], "eval_series":0, "cond_nr": 0}
    for result in jlist:
        #print("this is going ot be my dict: \n", result)
        dres= dict(result)
        if dres.keys() != standard_dict.keys():
            print(f"error keys dont match:\n {dres.keys()}\n vs. \n{standard_dict.keys()}")
            return 0
    if not return_res:
        with open(outpath, "w") as f:
            json.dump(jlist, f)
    else:
        return jlist

# list of condition lists comes in,
# file list comes in
# evaluates conditions on ht efiles and prints the result in  outfile.tsv, can be opened with libre office
def eval_pipeline(list_of_eval_sets, file_list, outfile):
    t0=time.time()
    res_col=[]
    conds_count=0
    for file in file_list:
        for num, eval_series in enumerate(list_of_eval_sets):
            print(f"Evaluating list {num+1} of {len(list_of_eval_sets)} on {file}")
            res_l=eval_conds(eval_series, file, max_ex=100, custom_id=True)
            for result in res_l:
                #adding a eval series number to the results from the same list of conditions (like nsubj list)
                result.update({"eval_series": num})
                result.update({"cond_nr": conds_count})
                res_col.append(result)
                conds_count+=1
    #print("this is the loop output:", res_col)
    evals=    json.dumps(print_eval_log(res_col, outfile, return_res=True))
    print("***SUCCESSFUL RUN***")
    print(f" {conds_count} evaluations done in {time.time()-t0}, av. time {(time.time()-t0)/conds_count} ")
    please=json.loads(evals)
    with open("eval/"+outfile, "w") as f:
        df=pd.DataFrame(please)
        df.to_csv(f, sep="\t", index=False)

    return 0

def merge_evals_old(condict1, condict2):
    mega_dict= {}
    for name1,cond1 in condict1.items():
        for name2, cond2 in condict2.items():
            l=[]
            for c1 in cond1:
                for c2 in cond2:
                    # unir condicions
                    z = c1 | c2
                    #print(z)
                    #si hi ha condicions incompatibles s hi afegeix una condicio impossible
                    for intersecting_keys in set(c1.keys()).intersection(set(c2.keys())):
                        if id(intersecting_keys) == id(has_sons):
                            #print(intersecting_keys, c1[intersecting_keys])
                            #print(intersecting_keys, c2[intersecting_keys])
                            z.update({has_sons : c1[intersecting_keys] + c2[intersecting_keys]})

                        if c1[intersecting_keys] != c2[intersecting_keys] and id(intersecting_keys) != id(has_sons):
                            #print("****non compatible conditions\n")
                            #print("condition 1\n:", c1[intersecting_keys])
                            #print("condition 2\n:",c2[intersecting_keys])
                            z = {"xpos":"impossible"}
                    l.append(z)
            mega_dict.update({name1 + "__und__" + name2 : l})

    for cross_name, cross_cond in mega_dict.items():
        print(cross_name, "::::", cross_cond)
    return mega_dict


#computes for each file all of the freqs of the different values it might have
# BUT it allows us to include a condition (since i  want to get rid of conditions as lists of dictionaries, since i only use them with one dictionary, it only works on the first element of the list :))

def conllu_file_freqs(filename, conllu_cat, conds=0, name_cond= "", rel=False):
    data = file_to_conllu(filename)
    d={}
    w=0
    d["filename"]= filename
    d["category"]= conllu_cat
    if not conds:
        d["conds"] = "-"
    else:
        if name_cond:
            d["conds"] = name_cond
        else:
            d["conds"] = conds[0]
    if rel:
        d["rel"]= 1
    else:
        d["rel"] = 0
    if not conllu_cat in data[0][0].keys():
        print(f"error {conllu_cat} is not a valid key.\n Valid keys are:",data[0][0].keys())
    else:
        if not conds:
            for sent in data:
               for j, token in enumerate(sent):
                    w+=1
                    #print(cat, token, type(token))
                    # I am turning token[cat] into a string bc when cat= feats it is a dictionary :/
                    t= token[conllu_cat]
                    t=str(t)
                    if t in d.keys():
                        d[t]+=1
                    else:
                        d[t]=1
        elif isinstance(conds[0], dict):
            for sent in data:
               for j, token in enumerate(sent):
                   if check_conds(sent, token, j, conds[0]):
                        w+=1
                        #print(cat, token, type(token))
                        # I am turning token[cat] into a string bc when cat= feats it is a dictionary :/
                        t= token[conllu_cat]
                        t=str(t)
                        if t in d.keys():
                            d[t]+=1
                        else:
                            d[t]=1
                   else:
                       pass
        else:
            print("ERR: conds[0] not a dict :-( ")
    if rel:
        for key, value in d.items():
            try:
                int(d[key])
            except:
                pass
            else:
                d[key] = 100*int(d[key])/w
        d["rel"]= w
    #print(d)
    return d
#computes for each file all of the freqs of the different values it might have
# but it allows us to include a condition (since i  want to get rid of conditions as lists of dictionaries, since i only use them with one dictionary, it only works on the first element of the list :))




# geta a conllu file list, it prints out the frequencies of all the different categories in different files
def file_freqs_eval(file_list):
    file_to_conllu(file_list[0])
    conllu_cats = file_to_conllu(file_list[0])[0][0].keys()
    print(conllu_cats)
    stupid_categories= ['id', 'misc', "head", "deps"]
    t0=time.time()
    for cat in conllu_cats:
        t1=time.time()
        print("We re starting with:", cat)
        if cat in stupid_categories:
            print(f"{cat} skipped")
            continue
        res_col=[]
        outfile=f"corpus_freqs_{cat}.csv"
        for file in file_list:
            #res_col.append(conllu_file_freqs(file, "upos", rel=False))
            res_col.append(conllu_file_freqs(file, cat, rel=True))
        #print("*"*16, cat, "*"*16, res_col)
       # check if the folder exists, else create it
        Path("eval/freq/").mkdir(parents=True, exist_ok=True)
        with open("eval/freq/"+outfile, "w") as f:
            df=pd.DataFrame(res_col)
            df.to_csv(f, sep=",", index=False)
        print(f"{cat} completed :) loop took {time.time()-t1}s, running for {time.time()-t0} seconds \n")
    return 0

# geta a conllu file list, it prints out the frequencies of all the different categories in different files
# and also an eval set, a list of dictionaries, pairs of a condition and its name
#prefix
def file_freqs_eval_filtered_by_conds(file_list, eval_set, prefix="filtered"):
    file_to_conllu(file_list[0])
    conllu_cats = file_to_conllu(file_list[0])[0][0].keys()
    print(conllu_cats)
    stupid_categories= ['id', 'misc', "head", "deps"]
    t0=time.time()
    for cat in conllu_cats:
        t1=time.time()
        print("We re starting with:", cat)
        if cat in stupid_categories:
            print(f"{cat} skipped")
            continue
        res_col=[]
        outfile=f"{prefix}_corpus_freqs_{cat}.csv"
        for file in file_list:
            d_res= conllu_file_freqs(file, cat, rel=False)
            d_res.update({ "cond_series": "none"})
            res_col.append(d_res)
            for i, condition in enumerate(eval_set):
                for keys, values in condition.items():
                    d_res= conllu_file_freqs(file, cat, conds= values, name_cond= keys, rel=False)
                    d_res.update({ "cond_series": i})
                    res_col.append(d_res)
        #print("*"*16, cat, "*"*16, res_col)
       # check if the folder exists, else create it
        Path("eval/freq/").mkdir(parents=True, exist_ok=True)
        with open("eval/freq/"+outfile, "w") as f:
            df=pd.DataFrame(res_col)
            df.to_csv(f, sep=",", index=False)
        print(f"{cat} completed :) loop took {time.time()-t1}s, running for {time.time()-t0} seconds \n")
    return 0







#returns a list of all the values for the conllu feateures present in the corpus
def conllu_possible_values():
    good_conllu_keys=["upos", "xpos", "feats", "deprel"]

    d_categories={}
    for cat in good_conllu_keys:
        d_categories[cat]=conllu_file_freqs(tv3_corpus, cat, rel=True)
        d_categories[cat].update(conllu_file_freqs(KI_corpus, cat, rel=True))
        del d_categories[cat]["filename"]
        del d_categories[cat]["category"]
        del d_categories[cat]["rel"]
        d_categories[cat]= list(d_categories[cat].keys())
        print(cat, d_categories[cat], "\n\n")
    return d_categories

In [41]:
# list of all the values in the conllu file generated w the function conllu_possible_values()
deprel_list = ['det', 'nsubj', 'amod', 'case', 'obj', 'flat', 'ROOT', 'mark', 'obl', 'compound', 'cc', 'conj', 'punct', 'appos', 'nmod', 'aux', 'ccomp', 'advmod', 'fixed', 'acl', 'nummod', 'cop', 'advcl', 'expl:pass', 'xcomp', 'csubj', 'iobj', 'parataxis', 'dep']
upos_list = ['DET', 'NOUN', 'ADJ', 'ADP', 'PROPN', 'VERB', 'SCONJ', 'NUM', 'CCONJ', 'PUNCT', 'AUX', 'SYM', 'PRON', 'ADV', 'INTJ', 'SPACE', 'PART']
upos_list_args= ['DET', 'NOUN', 'ADJ', 'PROPN','PRON', 'ADV','NUM', 'SYM']

#EVAL 1
# SVO in sentences of the right kind (sense complements verbals, sense que el verb sigui una subordinada -> pot ser coordinat)

# 1 AQUESTA ES LA CONDICIO D UN BON VERB (sense complements verbals, sense ser xcomp d un altre verb)
#  no ser el main verb d una subordinada vol dir restringir molts deprels:
# aux is also out bc it marks situations such as "cal marxar", "estan a punt d entrar"
# fixed is also aut bc it is basically "ès a dir"
# compound is also out bc it classifies constructions where verbs are very close to another "vol dir", "es fa dir"
# Evitar coordinacio amb subordinades -> per evitar frases com (1, '2025-10-10 00:33:22_habitatge_65_031_011')] de a1KI_corpus_65_habitatge_gemm.conllu
eval_v= {"verb": [{"upos": "VERB"}]}
eval_v_adequat = {"verb_ok": [{"upos": "VERB", is_not_any_of: [{"deprel": is_sub_clause_head}, {"deprel": "aux"}, {"deprel": "fixed"}, {"deprel": "compound"}, {"deprel": "conj", parent_is : {"upos":"VERB", "deprel" : is_sub_clause_head} }], nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}]  }] }

# this is to check how the deprels look like
eval_verb_deprels = {f"verb_{deprel}": [{"upos": "VERB", "deprel": deprel}]   for deprel in deprel_list}
eval_verb_derpel_no_subclause_arg = {f"verb_no_subclause_args_{deprel}": [{"upos": "VERB", "deprel": deprel, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}]}]   for deprel in deprel_list}

# ANEM A BUSCAR LA BOLA DE DRAC
# delimitem un verb amb un bon OBJ
# eliminem pronoms febles, eliminem complements preposicionals, eliminem pronoms relatius amb funcio d objecte,
# eliminem tambe deprel case per eliminar coses com ara: "com a mesures s han aplicat.."
conds_v_obj_good= [{"upos": "VERB", has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}]}]}]
eval_obj = {"verb_w_good_obj": conds_v_obj_good}


# ANEM A BUSCAR EL SUBJECTE DEL DRAC!
#
conds_verb_wsubj_true = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }]
eval_subj = {"subj_bo": conds_verb_wsubj_true}
# this is the list of evaluations for the simple presence of S V O in "good" sentences, this is, verbal predicates with no clausal core arguments which are also not the main verb of a subordinate clause
list_evals_presence_SVO = [eval_v, eval_v_adequat, merge_evals(eval_v_adequat, eval_subj), merge_evals(eval_v_adequat, eval_obj), merge_evals(eval_v_adequat, merge_evals(eval_subj, eval_obj)), ]

# EVAL 1.5
# what is the upos class ob our beautiful verb arguments?
eval_obj_upos= {f"verb__obj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos": upos ,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}]}]}] for upos in upos_list_args}
eval_subj_upos= {f"verb_subj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "upos":upos,  is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }] for upos in upos_list_args}
list_evals_SVO_type= [eval_v, eval_obj, eval_subj, eval_subj_upos, merge_evals(eval_subj, eval_obj),eval_obj_upos, merge_evals(eval_subj_upos, eval_obj_upos)]



#EVAL 2_
# order of the components in EVAL 1
# successful, beautifil
# order of the object: OV vs VO
conds_v_obj_good_OV = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}]}]}]
conds_v_obj_good_VO = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}]}]}]
conds_v_obj_good_OVO = [{"upos": "VERB", has_sons: [{"deprel": "obj", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}  ]}, {"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}  ]} ]}]

eval_OVO = {"obj_OV": conds_v_obj_good_OV, "obj_VO": conds_v_obj_good_VO, "obj_OVO": conds_v_obj_good_OVO}

# order of the subject
conds_v_subj_good_SV = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_VS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_SVS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }, {"deprel": "nsubj", is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]

eval_SVS= {"subj_SV": conds_v_subj_good_SV, "subj_VS": conds_v_subj_good_VS, "subj_SVS": conds_v_subj_good_SVS}

list_eval_SVO_order= list_evals_presence_SVO + [merge_evals(eval_v_adequat ,eval_OVO)] + [merge_evals(eval_v_adequat ,eval_SVS)] + [merge_evals(eval_v_adequat,merge_evals(eval_OVO, eval_SVS))]







# EVAL 3
# que passa amb les subordinades?
# CONDICIONS PER UNA SUBORDINADA DE FUNCIO X
# restringim a les que tenen un nexe de tipus SCONJ, q te deprel mark -> que no fa cap funcio
list_sub_deprels = ["csubj", "ccomp", "advcl", "acl", "xcomp"]

eval_v_sub_types = {f"v_sub_deprel_{deprel}": [{"upos": "VERB", "deprel": deprel, has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

eval_v_sub_types_coord = {f"v_sub_coord_deprel_{deprel}": [{"upos": "VERB", "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

# EVAL SVO presence in subordinates and coordinates with subordinates
list_evals_SVO_presence_subordinates = [eval_v, eval_v_sub_types, merge_evals(eval_v_sub_types, eval_subj), merge_evals(eval_v_sub_types, eval_obj), merge_evals(eval_v_sub_types, merge_evals(eval_subj, eval_obj)), ]
list_evals_SVO_presence_sub_coord = [eval_v, eval_v_sub_types_coord, merge_evals(eval_v_sub_types_coord, eval_subj), merge_evals(eval_v_sub_types_coord, eval_obj), merge_evals(eval_v_sub_types_coord, merge_evals(eval_subj, eval_obj)), ]

list_evals_SVO_presence_sub_both = list_evals_SVO_presence_subordinates + list_evals_SVO_presence_sub_coord

#EVAL SVO order in subordinates and coordinates w subordinates
list_eval_SVO_order_subordinates = list_evals_SVO_presence_subordinates + [merge_evals(eval_v_sub_types ,eval_OVO)] + [merge_evals(eval_v_sub_types ,eval_SVS)] + [merge_evals(eval_v_sub_types, merge_evals(eval_OVO, eval_SVS))]
list_eval_SVO_order_sub_coord = list_evals_SVO_presence_sub_coord + [merge_evals(eval_v_sub_types_coord ,eval_OVO)] + [merge_evals(eval_v_sub_types_coord ,eval_SVS)] + [merge_evals(eval_v_sub_types_coord, merge_evals(eval_OVO, eval_SVS))]

list_evals_SVO_order_sub_both = list_eval_SVO_order_subordinates + list_eval_SVO_order_sub_coord

# EVAL 3.5
# tipus de subordinades mes comprimit
# en comptes de tenir un nexe "SCONJ" (que, si,) que passa amb les q tenen nexe "ADP" (cansats de compartir, per destituir el president..)
# i a mes el verb esta en infinitiu:
eval_v_sub_types3 = {f"v_sub2_infinitiu_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": deprel, has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
eval_v_sub_types3_coord = {f"v_sub_infinitiu_coord_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
list_evals_subs_SVO_presence = [eval_v_sub_types3, eval_v_sub_types3_coord, merge_evals(eval_v_sub_types3, eval_obj), merge_evals(eval_v_sub_types3, eval_subj)]






# EVAL 4
# NOUN PHRASE WHATS UP
# idea: upos based
# nur nominale sintagmas die nur nomen haben

nom_deprels = ["nmod","appos", "nummod","acl", "amod", "det", "clf", "case"]
#general noun conditions: maybe i add more
conds_noun_ok= [{"upos":"NOUN"}]
eval_noun= {"eval_n": conds_noun_ok}

# DET ORDER
conds_n_det = [{"upos": "NOUN", has_sons: [{"upos": "DET", "head": head_smaller_than_id, is_not: {"deprel":"conj"}, has_no_sons: [{"deprel": "mark"}, {"deprel": "appos"}]}]}]
# tODO: check if det_n can have problems similar to the characterisation of n_det
conds_det_n= [{"upos": "NOUN", has_sons: [{"upos": "DET", "head": head_greater_than_id}]}]
conds_det_n_check= [{"upos": "NOUN", has_sons: [  {"upos": "DET", "head": head_greater_than_id, is_not: {"deprel":"conj"}, has_no_sons: [{"deprel": "mark"}, {"deprel": "appos"}]}] }]

conds_det_n_det = [merge_conditions_dicts(conds_det_n[0], conds_n_det[0])]

eval_det_n = {"det_n": conds_det_n, "det_n_check" : conds_det_n_check, "n_det":conds_n_det,  "det_n_det":conds_det_n_det}

# ADJ + N order
conds_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_greater_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_n_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_smaller_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_adj_n_adj= merge_conditions_dicts(conds_n_adj[0], conds_adj_n[0])

eval_adj_n = {"adj_n": conds_adj_n, "n_adj":conds_n_adj, "adj_n_adj":[conds_adj_n_adj]}

# ADV + ADJ
conds_adv_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_greater_than_id,  has_sons: [{"upos": "ADV"}] }]}]
conds_n_adv_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_smaller_than_id, has_sons: [{"upos": "ADV"}] }]}]
conds_adv_adj_n_adj= [merge_conditions_dicts(conds_n_adv_adj[0], conds_adv_adj_n[0])]

eval_adv_adj_n = {"adv_adj_n": conds_adv_adj_n, "n_adv_adj": conds_n_adv_adj, "n_adv_adj_beide":conds_adv_adj_n_adj}



# EVAL 5:
# numerals
conds_n_num= [{"upos": "NOUN", has_sons: [{"deprel": "nummod"}]}]
conds_n_num_upos= [{"upos": "NOUN", has_sons: [{"upos": "NUM"}]}]
conds_n_num2 = [{"upos":"NOUN", has_sons: [{"upos": "NUM", is_not: {"deprel": "nummod"} } ] }]
conds_n_num3 = [{"upos":"NOUN", has_sons: [{"deprel": "nummod", is_not: {"upos": "NUM"} } ] }]

eval_n_num ={"n_nummod": conds_n_num, "n_num_upos": conds_n_num_upos, "num_not_nummod": conds_n_num2, "nummod_not_num": conds_n_num3 }

list_nom_evals= [eval_noun, eval_det_n, eval_adj_n, eval_adv_adj_n, merge_evals(eval_det_n, eval_adj_n), merge_evals(eval_det_n, eval_adv_adj_n), ]


# EVAL: feats dels adverbis q acompanyen adjectius
conds_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_greater_than_id, parent_is: {"upos": "NOUN"}}}]
conds_n_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_smaller_than_id, parent_is: {"upos": "NOUN"}}}]

eval_adv_adj = {"adv_adj_n": conds_adv_adj, "n_adv_adj": conds_n_adv_adj}
adv_possible_feats= [{'Polarity': 'Neg'},{'Degree': 'Cmp'}]

eval_adv_types= {f"feat{feature}": [{feats_include: feature}] for feature in adv_possible_feats}
llista_adv_types = [eval_adv_adj, merge_evals(eval_adv_types, eval_adv_adj)]





merging:
 {'verb_ok': [{'upos': 'VERB', <function is_not_any_of at 0x7f874410e200>: [{'deprel': <function is_sub_clause_head at 0x7f874410ea20>}, {'deprel': 'aux'}, {'deprel': 'fixed'}, {'deprel': 'compound'}, {'deprel': 'conj', <function parent_is at 0x7f874410e0c0>: {'upos': 'VERB', 'deprel': <function is_sub_clause_head at 0x7f874410ea20>}}], <function has_sons at 0x7f873bcdb9c0>: [{'deprel': <function is_sub_clause_head_core_arg at 0x7f874410e980>}]}]}
 and:
 {'subj_bo': [{'upos': 'VERB', <function has_sons at 0x7f874410e340>: [{'deprel': 'nsubj', <function is_not at 0x7f874410e160>: {<function feats_include at 0x7f874410e020>: {'PronType': 'Rel'}}, <function has_no_son at 0x7f874410e5c0>: {'deprel': 'cop'}}]}]}
intersecting keys: {'upos'}
adding non intersecting keys
<function is_not_any_of at 0x7f874410e200> [{'deprel': <function is_sub_clause_head at 0x7f874410ea20>}, {'deprel': 'aux'}, {'deprel': 'fixed'}, {'deprel': 'compound'}, {'deprel': 'conj', <function parent_is at 0x7f87

In [42]:
 codi = "b2"
 #corpus
tv3_corpus=f"{codi}tv3_corpus.conllu"
KI_corpus=f"{codi}KI_corpus.conllu"

tv3_small= f"{codi}tv3_corpus_8_russia.conllu"
KI_small= f"{codi}KI_corpus_8_russia_gemm.conllu"
tv3_mid= f"{codi}tv3_corpus_70_habitatge.conllu"
KI_mid= f"{codi}KI_corpus_70_habitatge_gemm.conllu"

#some handy corpus lists:
file_list=[tv3_corpus, KI_corpus]
small_file_list=[tv3_small, KI_small]
mid_file_list=[tv3_mid, KI_mid]

In [43]:


eval_pipeline(list_evals_SVO_type, mid_file_list, "mid_eval_SVO_types.csv")

Evaluating list 1 of 7 on b1tv3_corpus_70_habitatge.conllu
	Evaluating verb on b1tv3_corpus_70_habitatge.conllu
	 	Evaluated verb on b1tv3_corpus_70_habitatge.conllu, it took 0.7304279804229736
done with list, it took 0.7305154800415039
Evaluating list 2 of 7 on b1tv3_corpus_70_habitatge.conllu
	Evaluating verb_w_good_obj on b1tv3_corpus_70_habitatge.conllu
	 	Evaluated verb_w_good_obj on b1tv3_corpus_70_habitatge.conllu, it took 0.9031083583831787
done with list, it took 0.9032042026519775
Evaluating list 3 of 7 on b1tv3_corpus_70_habitatge.conllu
	Evaluating subj_bo on b1tv3_corpus_70_habitatge.conllu
	 	Evaluated subj_bo on b1tv3_corpus_70_habitatge.conllu, it took 0.8438434600830078
done with list, it took 0.8439457416534424
Evaluating list 4 of 7 on b1tv3_corpus_70_habitatge.conllu
	Evaluating verb_subj_DET on b1tv3_corpus_70_habitatge.conllu
	 	Evaluated verb_subj_DET on b1tv3_corpus_70_habitatge.conllu, it took 0.8549811840057373
	Evaluating verb_subj_NOUN on b1tv3_corpus_70_hab

0

In [50]:
# codeblock to merge all files into one fat file :)
import glob
llista=["estatsunits","incendis","migracions", "educacio", "musica", "salut", "erc", "meteorologia", "incendis", "psoe", "habitatge", "russia", "israel", "ucraina"]
type= "KI"
llista_arxius=[]
for tema in llista:
    print("adding", tema)
    if not glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu"):
        raise Exception(f"File for topic {codi}{type}_corpus_*_{tema}*.conllu not found.")
    elif len(glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu")) > 0:
        glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu").sort(reverse=True)
        corpuspath = glob.glob(f"analysed_corpus/{codi}{type}_corpus_*_{tema}*.conllu")[0]
        corpusfile = corpuspath.split("/")[-1]
    llista_arxius.append(corpusfile)
conllu_merger(llista_arxius, f"{type}_corpus.conllu")


adding estatsunits
adding incendis
adding migracions
adding educacio
adding musica
adding salut
adding erc
adding meteorologia
adding incendis
adding psoe
adding habitatge
adding russia
adding israel
adding ucraina


In [1]:
print(llista_arxius)

#eval_pipeline(llista_adv_types, mid_file_list, "mid_adv_types_problem.csv")

NameError: name 'llista_arxius' is not defined

In [52]:
conllu_file_stats("KI_corpus.conllu")


{'words': 141859,
 'sentences': 5877,
 'av. sentence length': 24.137995575974138,
 'av. tree height': 4.71073677046112,
 'max tree height': 12,
 'tree height / sent length': 0.21903233708647898}